# 01 — Inference Demo
**On the Calibration of Large Language Models and Alignment**

Authors: Shahab Ahmad, Inam Ul Hassan, Ejaz Ulhaq  
Course: NLP — MS AI, FAST NUCES Islamabad

---
This notebook demonstrates:
1. Loading the Dolly LoRA fine-tuned model
2. Running inference on sample prompts
3. Computing ECE on a small subset
4. Plotting a reliability diagram


In [ ]:
# Install dependencies (run once)
# !pip install transformers peft datasets torch scipy matplotlib -q

In [ ]:
import sys, os
sys.path.append('..')  # so we can import from src/

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json

from src.utils import compute_ece, plot_reliability_diagram, build_mmlu_prompt, extract_mmlu_answer

print('Libraries loaded successfully')

## 1. Load Results from Saved Metrics

In [ ]:
with open('../results/baseline_metrics.json') as f:
    baseline = json.load(f)

with open('../results/improved_metrics.json') as f:
    improved = json.load(f)

print('Dolly LoRA results:')
print(json.dumps(improved['results'], indent=2))

## 2. ECE Comparison Table

In [ ]:
data = {
    'Model': ['Baseline (LLaMA-7B)', 'Alpaca LoRA (ep3)', 'OA LoRA (ep3)', 'Dolly LoRA (ep3)'],
    'ECE (CLM)':   [0.0092, 0.0856, 0.0262, 0.0201],
    'ACC (CLM)':   [0.6140, 0.5079, 0.5880, 0.6084],
    'ECE (Facts)': [0.0602, 0.2323, 0.0813, 0.0849],
    'ACC (Facts)': [0.1288, 0.0462, 0.0974, 0.1224],
    'ECE (MMLU)':  [0.0497, None,   0.0705, 0.0237],
    'ACC (MMLU)':  [0.4460, None,   0.4310, 0.2263],
}

df = pd.DataFrame(data).set_index('Model')
df.style.highlight_min(subset=['ECE (CLM)', 'ECE (Facts)', 'ECE (MMLU)'], color='lightgreen') \
        .highlight_max(subset=['ACC (CLM)', 'ACC (MMLU)'], color='lightyellow')

## 3. ECE Bar Chart — All Models & Tasks

In [ ]:
models = ['Baseline', 'Alpaca LoRA', 'OA LoRA', 'Dolly LoRA']
clm_ece   = [0.0092, 0.0856, 0.0262, 0.0201]
facts_ece = [0.0602, 0.2323, 0.0813, 0.0849]
mmlu_ece  = [0.0497, None,   0.0705, 0.0237]

x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width, clm_ece,   width, label='CLM',   color='steelblue')
bars2 = ax.bar(x,         facts_ece, width, label='Facts', color='darkorange')
bars3 = ax.bar(x + width, [v if v else 0 for v in mmlu_ece],
               width, label='MMLU', color='forestgreen')

ax.set_xticks(x)
ax.set_xticklabels(models, fontsize=11)
ax.set_ylabel('Expected Calibration Error (ECE) ↓', fontsize=11)
ax.set_title('ECE Comparison: Baseline vs Instruction-Tuned Models (Epoch 3)', fontsize=13)
ax.legend()

# Annotate bars
for bar in [*bars1, *bars2, *bars3]:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.002, f'{h:.4f}',
                ha='center', va='bottom', fontsize=8)

ax.text(x[1] + width, 0.003, 'bug', ha='center', fontsize=8, color='red')
plt.tight_layout()
plt.savefig('../results/ece_comparison.png', dpi=150)
plt.show()

## 4. MMLU ECE Only — Dolly Beats the Baseline

In [ ]:
mmlu_models = ['Baseline', 'Alpaca\n(ep3, bug)', 'OA LoRA', 'Dolly LoRA']
mmlu_ece_vals = [0.0497, 0.0970, 0.0705, 0.0237]
colors = ['steelblue', 'lightgray', 'darkorange', 'forestgreen']

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(mmlu_models, mmlu_ece_vals, color=colors, width=0.5)
ax.axhline(y=0.0497, color='red', linestyle='--', linewidth=1.5, label='Baseline ECE')

for bar, val in zip(bars, mmlu_ece_vals):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.001, f'{val:.4f}',
            ha='center', fontsize=10, fontweight='bold')

ax.set_ylabel('MMLU ECE ↓', fontsize=12)
ax.set_title('MMLU Calibration — Dolly LoRA Achieves Best ECE\n(Falls Below Untuned Baseline)', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('../results/mmlu_ece_comparison.png', dpi=150)
plt.show()

## 5. Training Dynamics — Pythia-1B Calibration over Steps

In [ ]:
steps = [256, 512, 1000, 4000, 16000, 64000, 143000]
clm_ece_dyn   = [0.0774, 0.0498, 0.0656, 0.0473, 0.0456, 0.0457, 0.0420]
facts_ece_dyn = [0.1452, 0.1117, 0.0489, 0.0658, 0.0687, 0.0580, 0.0482]
mmlu_ece_dyn  = [0.5690, 0.4730, 0.3230, 0.2400, 0.2440, 0.2110, 0.1210]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(steps, clm_ece_dyn,   'o-', label='CLM ECE',   color='steelblue')
ax.plot(steps, facts_ece_dyn, 's-', label='Facts ECE', color='darkorange')
ax.plot(steps, mmlu_ece_dyn,  '^-', label='MMLU ECE',  color='forestgreen')
ax.set_xscale('log')
ax.set_xlabel('Training Step (log scale)', fontsize=11)
ax.set_ylabel('ECE ↓', fontsize=11)
ax.set_title('Pythia-1B: Calibration Improves with Training Steps', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/training_dynamics.png', dpi=150)
plt.show()

## 6. Simulated Reliability Diagram
Demonstrates ECE computation using synthetic confidence/label data matching our reported results.

In [ ]:
np.random.seed(42)
n = 3000

# Simulate Dolly LoRA MMLU predictions (ECE ~0.0237 — well-calibrated)
dolly_confs = np.random.beta(2, 5, n)  # modest confidence
dolly_labels = (np.random.rand(n) < dolly_confs).astype(int)

# Simulate Alpaca LoRA CLM predictions (ECE ~0.0856 — overconfident)
alpaca_confs = np.random.beta(8, 2, n)  # high confidence
alpaca_labels = (np.random.rand(n) < dolly_confs).astype(int)  # lower accuracy

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, confs, labels, title in [
    (axes[0], dolly_confs,  dolly_labels,  'Dolly LoRA (well-calibrated)'),
    (axes[1], alpaca_confs, alpaca_labels, 'Alpaca LoRA (overconfident)'),
]:
    ece = compute_ece(confs.tolist(), labels.tolist())
    bins = np.linspace(0, 1, 11)
    bin_accs, bin_confs = [], []
    for i in range(10):
        mask = (confs >= bins[i]) & (confs < bins[i+1])
        if mask.sum():
            bin_accs.append(labels[mask].mean())
            bin_confs.append(confs[mask].mean())
        else:
            bin_accs.append(0); bin_confs.append((bins[i]+bins[i+1])/2)
    centers = [(bins[i]+bins[i+1])/2 for i in range(10)]
    ax.plot([0,1],[0,1],'k--',label='Perfect')
    ax.bar(centers, bin_accs, width=0.08, alpha=0.6, color='steelblue', label='Accuracy')
    ax.set_xlim(0,1); ax.set_ylim(0,1)
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy')
    ax.set_title(f'{title}\nECE = {ece:.4f}')
    ax.legend()

plt.suptitle('Reliability Diagrams', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../results/reliability_diagrams.png', dpi=150)
plt.show()

## 7. Load Model for Live Inference (optional — requires weights)

Uncomment and run if you have trained weights in `../checkpoints/dolly_lora/final/`

In [ ]:
# from src.model import load_lora_model
#
# BASE_MODEL = "huggyllama/llama-7b"
# ADAPTER_PATH = "../checkpoints/dolly_lora/final"
#
# model, tokenizer = load_lora_model(BASE_MODEL, ADAPTER_PATH)
# model.eval()
#
# prompt = "### System:\nYou are a helpful assistant.\n\n### Instruction:\nWhat is calibration in AI?\n\n### Response:\n"
# inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
#
# with torch.no_grad():
#     output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
#
# print(tokenizer.decode(output[0], skip_special_tokens=True))